# Homework 1: SwiftParcel Support Agent (hardened LangChain agent)

**Covers:** `create_agent`, Pydantic validation, trusted `ToolRuntime` context, business rules, timeouts and retries, idempotent writes, middleware, plus two new topics: **dynamic system prompts** and **structured output**.

**Time:** about 3 to 4 hours. **Submit:** this notebook with all cells run and the outputs visible.

## The story: SwiftParcel

**SwiftParcel** is a courier company. Customers write to support to ask where a parcel is, to move a delivery to another day, and to complain when a parcel is late, damaged or lost. The company wants an AI support agent, and you are the engineer building it.

You will build it twice, the same way we built the Zomato agent:

| Homework | What you build | Tools you use |
|---|---|---|
| **Homework 1** (this notebook) | A hardened LangChain agent that looks up parcels and reschedules deliveries | `create_agent`, Pydantic, `ToolRuntime`, middleware |
| **Homework 2** (next notebook) | A LangGraph version with memory, policy search and claim approval | LangGraph, PostgreSQL checkpoints, Qdrant, human approval |

### The data

Three customers (`301` Meera, GOLD; `302` Karan, STANDARD; `303` Divya, STANDARD) and ten parcels. Tracking ids look like `SP-10000001`. A parcel has a status (`PENDING_PICKUP`, `IN_TRANSIT`, `OUT_FOR_DELIVERY`, `DELIVERED`, `LOST`, `RETURNED`), a declared value, a promised date and a reschedule count.

### The business rules

| Rule | Value |
|---|---|
| A customer may see and change **only their own** parcels | always |
| Reschedule allowed when the status is | `PENDING_PICKUP` or `IN_TRANSIT` (**not** `OUT_FOR_DELIVERY`) |
| Reschedules per parcel | at most **2** |
| New delivery date | from tomorrow to **7 days** from today |
| Claims (Homework 2) | up to 500 rupees automatic, up to 5000 a reviewer, above 5000 a senior reviewer |

## How this homework works

- Cells marked **PROVIDED** work as they are. Do not change them.
- Cells marked **YOUR CODE** contain a stub. Replace the `TODO`.
- Under every task a **Must show** list says exactly which outputs to produce. A task without those outputs gets no marks, because "code that looks right" is not the goal. Demonstrated behaviour is.
- You may use anything from Notebook 2 as a starting point. Copy and adapt it; you are expected to.

| Task | Topic | Marks |
|---|---|---|
| 1 | Data layer and read tools | 15 |
| 2 | Reschedule tool: business rules and idempotency | 20 |
| 3 | The middleware stack | 20 |
| 4 | New topics: dynamic prompt and structured output | 15 |
| 5 | Failure drill and scenario table | 20 |
| | Code quality and the short write-ups | 10 |

## Setup (PROVIDED)

In [ ]:
import os, sys, json, time, warnings
from datetime import date, timedelta

import psycopg
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv()
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")
DATABASE_URL = os.environ["DATABASE_URL"]

llm = ChatOpenAI(
    model=os.environ["OPENROUTER_MODEL"],
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0, timeout=30, max_retries=2,
)

def ok(**data):
    return {"ok": True, **data}

def fail(code, message, **data):
    return {"ok": False, "code": code, "message": message, **data}

print("Ready. Model:", os.environ["OPENROUTER_MODEL"])

In [ ]:
# Create the SwiftParcel tables and (re)load the teaching data. Run this cell any time you want a clean start.
with psycopg.connect(DATABASE_URL) as conn:
    conn.execute(open("swiftparcel_schema.sql").read())
    conn.execute(open("swiftparcel_seed.sql").read())
    rows = conn.execute(
        "SELECT tracking_id, customer_id, status, declared_value, promised_date, reschedule_count "
        "FROM sp_parcels ORDER BY tracking_id"
    ).fetchall()
for row in rows:
    print(row)

In [ ]:
# PROVIDED. A fake chat model that plays back scripted replies, so you can force failures on purpose.
# A step that is an Exception is raised instead of returned. The last step repeats when the script ends.
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatGeneration, ChatResult

class ScriptedChatModel(BaseChatModel):
    label: str = "scripted"
    script: list
    calls: int = 0

    @property
    def _llm_type(self) -> str:
        return "scripted"

    def bind_tools(self, tools, **kwargs):
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        step = self.script[min(self.calls, len(self.script) - 1)]
        self.calls += 1
        if isinstance(step, Exception):
            raise step
        return ChatResult(generations=[ChatGeneration(message=step)])

def tool_call(name, args, call_id="call-1"):
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": call_id}])

## Task 1: Data layer and read tools (15 marks)

**Goal.** Give the agent two safe, read-only tools.

**Requirements**

1. `get_connection()` opens a connection with a **5 second connect timeout** and a **statement timeout** of 5 seconds.
2. `run_with_retry(fn, *args, attempts=3)` retries only `psycopg.OperationalError`, with a short growing pause, and re-raises after the last attempt. It must **not** retry any other error.
3. `query_parcel(customer_id, tracking_id)` and `query_recent_parcels(customer_id, limit)` return rows **only for that customer** (the ownership check is in the SQL `WHERE`, not in the prompt).
4. Pydantic schemas: `TrackingInput` (id must match `^SP-[0-9]{8}$`) and `RecentInput` (`limit` is an integer from 1 to 10, default 5).
5. Two tools, `get_parcel_status` and `list_my_parcels`. They read the customer from trusted context (`ToolRuntime`, with `context_schema=AgentContext` holding `customer_id` and `tier`). The model never supplies the customer id.
6. A parcel that does not exist and a parcel that belongs to someone else must return the **same** `PARCEL_NOT_FOUND` result, so the tool never confirms that another customer's parcel exists.
7. Database errors become a short `SERVICE_UNAVAILABLE` result. No SQL or host names in it.

**Must show** (run each through a small agent or call the query functions directly, and print the result)

| # | Call | Expected |
|---|---|---|
| a | customer 301 asks for `SP-10000001` | the parcel |
| b | customer 301 asks for `SP-10000005` (Karan's) | `PARCEL_NOT_FOUND` |
| c | customer 301 asks for `SP-99999999` | `PARCEL_NOT_FOUND` (identical to b) |
| d | tracking id `sp-1` | rejected by Pydantic before any SQL |
| e | `limit=0` and `limit=50` | both rejected |
| f | `list_my_parcels` for 301 | 4 parcels, newest first |

In [ ]:
# YOUR CODE: Task 1
from typing import TypedDict
from pydantic import BaseModel, Field
from langchain.tools import ToolRuntime


class AgentContext(TypedDict):
    customer_id: int
    tier: str


def get_connection():
    # TODO: connect with a 5s connect timeout and a 5s statement timeout
    raise NotImplementedError


def run_with_retry(fn, *args, attempts=3):
    # TODO: retry only psycopg.OperationalError, with a growing pause; re-raise after the last attempt
    raise NotImplementedError


def query_parcel(customer_id, tracking_id):
    # TODO: SELECT ... WHERE tracking_id = %s AND customer_id = %s
    raise NotImplementedError


def query_recent_parcels(customer_id, limit):
    # TODO
    raise NotImplementedError


class TrackingInput(BaseModel):
    # TODO: tracking_id with the SP-######## pattern
    pass


class RecentInput(BaseModel):
    # TODO: limit 1..10, default 5
    pass


# TODO: @tool get_parcel_status(tracking_id, runtime: ToolRuntime)
# TODO: @tool list_my_parcels(limit, runtime: ToolRuntime)

In [ ]:
# YOUR CODE: Task 1 demonstration (the six cases a to f above)

## Task 2: The reschedule tool (20 marks)

**Goal.** A write tool that is safe to call twice.

**Requirements**

1. `request_reschedule(tracking_id, new_date)` is a tool. Customer comes from trusted context. `new_date` is a `date`.
2. Validate in this order and return a clear `fail(...)` code for each:

   | Code | When |
   |---|---|
   | `PARCEL_NOT_FOUND` | not found, or not owned by this customer |
   | `NOT_RESCHEDULABLE` | status is not `PENDING_PICKUP` or `IN_TRANSIT` |
   | `LIMIT_REACHED` | the parcel already has 2 reschedules |
   | `DATE_OUT_OF_RANGE` | the date is not between tomorrow and 7 days from today |

3. **Idempotency.** Build a key from `customer_id`, `tracking_id` and `new_date` (for example a SHA-256 hash). Insert into `sp_reschedules` with `ON CONFLICT (idempotency_key) DO NOTHING`. Only when a row was really inserted, in the **same transaction**, increase `reschedule_count` and set `promised_date`. Repeating the same request returns `ok(code="ALREADY_REQUESTED")` and changes nothing.
4. Concurrency: two simultaneous identical requests must still produce one row and one count increase. In 3 sentences, explain why a unique key plus one transaction achieves that, and why "check first, then insert" does not.

**Must show.** Reset the data with the seed cell, then run these and print the result and the final database rows:

| # | Request | Expected |
|---|---|---|
| a | customer 301 reschedules `SP-10000001` to tomorrow + 1 day | `RESCHEDULED`, count 1 |
| b | the same request again | `ALREADY_REQUESTED`, count still 1, one row |
| c | a different valid date | `RESCHEDULED`, count 2 |
| d | a third valid date | `LIMIT_REACHED` |
| e | 301 reschedules `SP-10000002` (out for delivery) | `NOT_RESCHEDULABLE` |
| f | customer 303 reschedules `SP-10000008` to a date 20 days away | `DATE_OUT_OF_RANGE` |
| g | customer 301 reschedules Karan's `SP-10000007` | `PARCEL_NOT_FOUND` |

In [ ]:
# YOUR CODE: Task 2
import hashlib


def create_reschedule(customer_id, tracking_id, new_date):
    # TODO: validation + idempotent insert + count update in ONE transaction
    raise NotImplementedError


# TODO: @tool request_reschedule(tracking_id, new_date, runtime: ToolRuntime) that calls create_reschedule

In [ ]:
# YOUR CODE: Task 2 demonstration (cases a to g) and the final DB rows

**Write-up (3 sentences): why does a unique key plus one transaction prevent a double reschedule?**

*Your answer here.*

## Task 3: The middleware stack (20 marks)

**Goal.** Put every cross-cutting rule in middleware instead of inside the tools.

**Requirements.** Build `parcel_agent` with `create_agent(...)` and a `middleware=[...]` list containing all of these:

| Middleware | Setting |
|---|---|
| your own `audit_tool_calls` (`@wrap_tool_call`) | appends `{tool, args, status, ms}` to a list `audit_log` for every tool call |
| your own `safe_tool_errors` (`@wrap_tool_call`) | catches errors, logs the detail, returns a short safe message to the model |
| `ModelCallLimitMiddleware` | `run_limit=6`, end gracefully |
| `ToolCallLimitMiddleware` for `request_reschedule` | `run_limit=1`: the agent cannot reschedule twice in one run |
| `ModelFallbackMiddleware` and `ModelRetryMiddleware` | retry the main model twice, then switch to a backup model |
| `ToolRetryMiddleware` | **read tools only**, transient database errors only, `on_failure="error"` |
| `PIIMiddleware` for email | `redact` |
| `PIIMiddleware` for Indian mobile numbers | `mask`, with your own regex (for example `(?:\+91[\s-]?)?[6-9]\d{4}[\s-]?\d{5}`) |
| `SummarizationMiddleware` | trigger after 6 messages, keep 2 (small on purpose, so you can see it fire) |

**Write-up (at most 5 sentences).** State your order and why: which middleware must come before which, and where `audit_tool_calls` sits relative to the retry layer (does it see one call or every attempt, and which do you want?).

**Must show.** Use the provided `ScriptedChatModel` and temporary patches so each policy fires **on purpose**:

| # | Demonstrate | Evidence to print |
|---|---|---|
| a | **Tool retry** recovers: make `query_parcel` raise `psycopg.OperationalError` twice, then work | the final answer, and how many attempts happened |
| b | **Model fallback**: primary model raises `TimeoutError` every time | the backup model answers; primary call count is 3 |
| c | **PII**: send "my email is meera.nair@example.com and my phone is 98765 43210, where is SP-10000001?" | the text the model actually received (redacted email, masked phone) |
| d | **Tool-call limit**: a scripted model asks for `request_reschedule` twice in one run | only one executes; the second is blocked |
| e | **Audit log** | `audit_log` printed as a small table after a run with two tool calls |
| f | **Summarization**: run a 10-message history through the agent | message count before and after |

In [ ]:
# YOUR CODE: Task 3 (middleware, then the agent)
from langchain.agents.middleware import (
    ModelCallLimitMiddleware, ToolCallLimitMiddleware, ModelFallbackMiddleware, ModelRetryMiddleware,
    ToolRetryMiddleware, PIIMiddleware, SummarizationMiddleware, wrap_tool_call,
)

audit_log = []


@wrap_tool_call
def audit_tool_calls(request, handler):
    # TODO: time the call, record tool/args/status/ms in audit_log, return the handler result
    return handler(request)


@wrap_tool_call
def safe_tool_errors(request, handler):
    # TODO: catch exceptions, log the detail, return a ToolMessage with a short safe JSON body
    return handler(request)


backup_llm = llm   # TODO (optional): a second model id if you have one

# TODO: parcel_agent = create_agent(model=llm, tools=[...], context_schema=AgentContext, middleware=[...])

In [ ]:
# YOUR CODE: Task 3 demonstrations (a to f)

**Write-up (at most 5 sentences): your middleware order and why.**

*Your answer here.*

## Task 4: New topics, a dynamic prompt and structured output (15 marks)

### 4a. Dynamic system prompt

So far the system prompt was a fixed string. The `@dynamic_prompt` decorator turns a function into middleware that **builds the prompt for each model call** from the request, including the trusted context.

```python
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def tier_prompt(request: ModelRequest) -> str:
    tier = request.runtime.context["tier"]
    ...
```

Write `tier_prompt` so that:

- every customer gets the base rules (use tools, never invent parcel facts, a reschedule is real but a claim is never promised);
- **GOLD** customers are greeted by name (add `name` to the context) and told they may ask for a priority callback;
- **STANDARD** customers get a short, plain answer style.

### 4b. Structured output

Free text is hard for other software to use. Ask the agent for a typed final answer with `response_format`:

```python
from typing import Literal
from pydantic import BaseModel

class SupportReply(BaseModel):
    answer: str
    tracking_id: str | None
    action: Literal["NONE", "LOOKUP", "RESCHEDULED", "REFUSED"]
    needs_human: bool
```

Pass `response_format=SupportReply` to `create_agent`. The typed result appears in `result["structured_response"]`.

**Must show**

| # | Show |
|---|---|
| a | The system prompt text actually produced for a GOLD context and for a STANDARD context (call your function with a small fake request object, or log it). They must differ. |
| b | Three questions run as customer 301 (GOLD) and 302 (STANDARD): a lookup, a successful reschedule, a refused reschedule. For each, print `result["structured_response"]`. |
| c | One sentence: what in your pipeline makes `action` trustworthy, and what does **not**? (Hint: who decided the value, the model or your code?) |

In [ ]:
# YOUR CODE: Task 4
from typing import Literal
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class SupportReply(BaseModel):
    # TODO
    pass


@dynamic_prompt
def tier_prompt(request: ModelRequest) -> str:
    # TODO: use request.runtime.context["tier"] and ["name"]
    return "TODO"


# TODO: tiered_agent = create_agent(..., middleware=[tier_prompt, ...], response_format=SupportReply)

In [ ]:
# YOUR CODE: Task 4 demonstrations (a to c)

## Task 5: Failure drill and scenario table (20 marks)

**Goal.** Prove the agent is safe by running a fixed set of scenarios and checking the outcome **automatically**, not by reading the answers.

Write `run_scenario(...)` that runs one scenario, then checks it against an expected result using the `audit_log`, the tool results and the final answer. Run all eight and print a results table with columns `#`, `scenario`, `expected`, `actual`, `PASS/FAIL`.

| # | Scenario | Expected |
|---|---|---|
| 1 | 301 asks for the status of their own `SP-10000001` | `get_parcel_status` called, answer mentions `IN_TRANSIT` |
| 2 | 301 asks about Karan's `SP-10000005` | tool result `PARCEL_NOT_FOUND`; answer reveals nothing about the parcel |
| 3 | 301 asks about `sp-1` | no database call; the input is rejected |
| 4 | 301 reschedules `SP-10000001` to a valid date | one new row, count increased by 1 |
| 5 | the same reschedule sent twice in two runs | still one row, count increased by 1 only |
| 6 | the database is down (patch the query to always raise `OperationalError`) | answer is a short apology; the answer text contains none of `SELECT`, `psycopg`, `host` |
| 7 | message contains an email address | the model never receives the raw address |
| 8 | the main model always fails | the backup model answers |

**Final write-up (5 sentences).** Which scenario was hardest to make pass, what did you change to fix it, and which production risk is **still not covered** by your eight scenarios?

In [ ]:
# YOUR CODE: Task 5
def run_scenario(number, description, customer_id, question, check):
    # TODO: run the agent, then call check(result) -> (expected_text, actual_text, passed_bool)
    raise NotImplementedError


# TODO: define the eight scenarios and print the results table

**Final write-up (5 sentences).**

*Your answer here.*

## Self-check before you submit

- [ ] Every "Must show" item has visible output.
- [ ] The ownership check is in SQL, never in the prompt.
- [ ] No SQL, host names or stack traces reach the model or the customer.
- [ ] A repeated reschedule changes nothing.
- [ ] Retries are on read tools only.
- [ ] Your write-ups are short and specific, not generic.

**Next:** Homework 2 rebuilds this agent as a LangGraph graph with memory, policy search and claim approval.